In [1]:
# Imports
import sys
from pathlib import Path
import warnings

ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.pategan.models import PATEGAN

warnings.filterwarnings("ignore", message="X does not have valid feature names")

# ---------- Custom PATEGAN for CAR ----------
class PATEGANCar(PATEGAN):
    def __init__(self):
        super().__init__(
            epsilon=5.0,     # relaxed privacy → better scores
            delta=1e-5,
            num_teachers=5,  # fewer teachers → more data per teacher
            niter=8000,      # you can change to 1000/8000 later
            batch_size=64,
            learning_rate=5e-4,
            lambda_gp=5.0,
            random_state=42,
        )

# ---------------- Preprocess data (CAR) ----------------
dataset_path = ROOT / "raw_data" / "nursery.csv"
output_path = ROOT / "discretized_data" / "nursery.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path))

# ---------------- Run Train/Test/Synthetic pipeline ----------------
input_csv = str(output_path)          # discretised CAR data
output_dir = str(ROOT / "sample_data" / "nursery")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "nursery" / "pategan")

model_preview = PATEGANCar()
print("PATEGAN will train for:", model_preview.niter, "iterations")

pipeline = TrainTestSplitPipeline(model=PATEGANCar)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\nursery.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\nursery.csv
PATEGAN will train for: 8000 iterations
Loaded data with shape: (12960, 9)
Saved train/test full data
Train size: (10368, 9), Test size: (2592, 9)
Train label distribution:
 8
0    0.333333
1    0.329186
3    0.312018
4    0.025270
2    0.000193
Name: proportion, dtype: float64
Test label distribution:
 8
0    0.333333
1    0.329090
3    0.312114
4    0.025463
Name: proportion, dtype: float64
Saved X/y split
Training shape: (10368, 8) (10368,)
Test shape: (2592, 8) (2592,)
Remapped y classes: [0, 1, 2, 3, 4] -> [0, 1, 2, 3, 4]
Loaded training data: X shape=(10368, 8), y shape=(10368,)

Instructions for updating:
non-resource variables are not supported in the long term

Training PATE-GAN with ε=5.0, δ=1e-05
Teachers: 5, Iterations: 8000


Training: 100%|██████████| 8000/8000 [01:33<00:00, 85.46it/s] 


Training completed!

Generating 10368 synthetic samples...
[PATEGAN] Adding 4 dummy samples to cover classes: [0, 1, 2, 4]
Remapped y_test.csv to match synthetic data encoding
Saved x_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\pategan\x_synth.csv
Saved y_synth.csv to C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\pategan\y_synth.csv
Saved metadata.json to C:\Users\Prabu\Downloads\Katabatic\synthetic\nursery\pategan\metadata.json


C:\Users\Prabu\AppData\Local\pypoetry\Cache\virtualenvs\katabatic-AJr3_ocI-py3.11\Lib\site-packages\xgboost\training.py:199: UserWarning: [21:02:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Results saved to: Results\nursery\pategan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.2990
F1 Score: 0.2090

MLP:
Accuracy: 0.3241
F1 Score: 0.3064

RF:
Accuracy: 0.3171
F1 Score: 0.3362

XGBoost:
Accuracy: 0.4259
F1 Score: 0.3782
Train test split pipeline executed successfully.
